<a href="https://colab.research.google.com/github/boss-defender/Born-Baby-Ai/blob/main/Just_Born_Baby_Ai_for_gguf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 👶 Universal Baby AI: GGUF & LM Studio / Ollama Aligned Edition

A production Deep Learning framework implementing the **DeepSeek-V2/V3/R1 Architecture (`deepseek2`)**, engineered for 100% out-of-the-box compatibility with **`llama.cpp`**, **LM Studio**, and **Ollama**:
- **Native DeepSeek Causal Backbone**: Multi-Head Latent Attention (MLA) with low-rank KV compression ($c_t^{KV}$), Decoupled RoPE, and DeepSeekMoE (Top-2 Routed Experts + Shared Foundation Expert).
- **100% GGUF & LM Studio / Ollama Compatible**: Standardized `deepseek2` tensor naming and config metadata allowing instant conversion and drag-and-drop inference.
- **Thinking & Reasoning Engine**: Native DeepSeek-R1 style `<think>` chain-of-thought tokens.
- **Elastic Multi-Scale Presets**: Scale from `Baby-Nano` (~350K params) up to `Baby-Pro` (~180M params) or full `Custom` parameters.
- **Universal Domain Ingestion**: Automatic classification for **General Chat**, **Coding**, **Mathematics/Reasoning**, **Science/STEM**, and **Raw Pretraining**.
- **Production Release Packaging**: Exports `model.safetensors`, `config.json`, and pre-compiled `baby_ai.gguf` + `Modelfile` directly into `./baby_ai_model/`.

# **⚠️ Caution:**
**📢 1. With colab free tier , you can train with smaller datasets or lower max samples.**

**‼️ 2. For different datasets , you may need to make minor edits to the code (specially cell 2).**

**🚩 3. Make sure to give correct max-samples integer or leave it empty.**

In [ ]:
# @title ⚙️ Cell 1: Dependencies, Automated Drive Mounting & Universal UI Form { run: "auto" }
# @markdown Configure your model scale, task domain, dataset, and training parameters using the visual form below.

# @markdown ### 🧠 Model Identifier & Elastic Capacity
MODEL_SCALE = "Baby-Flash"  # @param ["Baby-Nano", "Baby-Tiny", "Baby-Flash", "Baby-Pro", "Custom"]
CUSTOM_HIDDEN_SIZE = 256  # @param {type:"integer"}
CUSTOM_NUM_LAYERS = 6  # @param {type:"integer"}
CUSTOM_NUM_EXPERTS = 8  # @param {type:"integer"}

# @markdown ### 🎯 Task Domain & Specialization
TASK_DOMAIN = "auto-detect"  # @param ["auto-detect", "general_chat", "coding", "mathematics_reasoning", "science_stem", "vision", "raw_pretrain"]

# @markdown ### 📂 Dataset Source & Location
DATASET_SOURCE = "Hugging Face"  # @param ["Hugging Face", "Custom Upload / Drive"]
DATASET_PATH = "Akhil391/daily_dialog"  # @param {type:"string"}
MAX_SAMPLES = None  # @param {type:"integer"}
MAX_SEQ_LENGTH = 128  # @param {type:"integer"}

# @markdown ### 💾 Google Drive Smart Checkpointing (Training State Only)
SAVE_TO_DRIVE = True  # @param {type:"boolean"}
SAVE_EVERY_N_STEPS = 200  # @param {type:"integer"}
MAX_CHECKPOINTS_TO_KEEP = 3  # @param {type:"integer"}

# @markdown ### 📦 Local Export Directory (Hugging Face / GGUF Standard Package)
LOCAL_EXPORT_DIR = "./baby_ai_model"  # @param {type:"string"}

# @markdown ### 🚀 Scalable Training Controls
BATCH_SIZE = 8  # @param [2, 4, 8, 16, 32]
GRAD_ACCUM_STEPS = 1  # @param [1, 2, 4, 8, 16] - Simulate large batches without OOM
EPOCHS = 3  # @param {type:"integer"}
LEARNING_RATE = 3e-4  # @param {type:"number"}
ENABLE_MIXED_PRECISION = True  # @param {type:"boolean"}

# @markdown ### 🌐 Hugging Face Hub (Optional Publishing)
PUSH_TO_HUB = False  # @param {type:"boolean"}
HF_TOKEN = ""  # @param {type:"string"}
HF_REPO_ID = "my-babyflash-model"  # @param {type:"string"}

# 1. Silent Automated Dependency Installation
print("Installing core dependencies (silently)...")
import subprocess
import sys
import os

deps = ["transformers", "datasets", "accelerate", "sentencepiece", "pillow", "einops", "safetensors", "gguf"]
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + deps)
    print("✓ Dependencies verified (transformers, datasets, accelerate, safetensors, gguf).")
except Exception as e:
    print(f"Notice: Dependency installation message: {e}")

# 2. Automated Google Drive Mount Engine (Training Resume State Only)
BASE_CHECKPOINT_DIR = "./BabyAI_Checkpoints"
if SAVE_TO_DRIVE:
    try:
        if 'google.colab' in sys.modules or os.path.exists('/content'):
            from google.colab import drive
            drive.mount('/content/drive')
            BASE_CHECKPOINT_DIR = "/content/drive/MyDrive/BabyAI_Checkpoints"
            print("✓ Google Drive mounted at /content/drive")
            print(f"✓ Training Checkpoints (Resume State) will sync to: {BASE_CHECKPOINT_DIR}")
        else:
            print("Notice: Local environment detected. Checkpoints will save to ./BabyAI_Checkpoints")
    except Exception as e:
        print(f"Notice: Google Drive mount skipped ({e}). Checkpoints save to ./BabyAI_Checkpoints")

os.makedirs(BASE_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOCAL_EXPORT_DIR, exist_ok=True)

# 3. Hardware Audit
import torch
print("=" * 60)
print(f"PyTorch Version: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Active Compute Device: {device.upper()}")
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Hardware: {torch.cuda.get_device_name(0)} ({vram_gb:.2f} GB VRAM)")
    torch.cuda.empty_cache()
else:
    print("Mode: CPU (Low-memory guardrails and gradient checkpoints active)")
print(f"Configured Scale: {MODEL_SCALE} | GGUF Alignment: DeepSeek-V2/V3 (deepseek2)")
print("=" * 60)


In [ ]:
# @title 📊 Cell 2: Universal Data Ingestion & Domain Auto-Classifier
Run_This_Cell= "2" # @param {type:"string"}
import os
import re
from typing import Dict, Any, List, Tuple
from datasets import load_dataset
from transformers import AutoTokenizer

# 1. Directory Sanitization & Unique Run Identification
def sanitize_identifier(name: str) -> str:
    cleaned = re.sub(r'[^a-zA-Z0-9_]', '_', str(name).strip())
    cleaned = re.sub(r'_+', '_', cleaned).strip('_')
    return cleaned or "unnamed"

UNIQUE_RUN_ID = f"{sanitize_identifier(MODEL_SCALE)}_on_{sanitize_identifier(DATASET_PATH)}"
RUN_CHECKPOINT_DIR = os.path.join(BASE_CHECKPOINT_DIR, UNIQUE_RUN_ID)
os.makedirs(RUN_CHECKPOINT_DIR, exist_ok=True)

print("=" * 60)
print(f"✓ Unique Training Run ID: '{UNIQUE_RUN_ID}'")
print(f"✓ Drive Checkpoint Folder (Resume Only): '{RUN_CHECKPOINT_DIR}'")
print(f"✓ Local Release Package Folder: '{LOCAL_EXPORT_DIR}'")
print("=" * 60)

# 2. Tokenizer Setup with Standard ChatML, Reasoning & Special Tokens
TOKENIZER_NAME = "Qwen/Qwen2.5-0.5B"
print(f"Loading tokenizer: {TOKENIZER_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

SPECIAL_TOKENS = {
    "additional_special_tokens": [
        "<|im_start|>", "<|im_end|>",              # ChatML delimiters
        "<think>", "</think>",                      # DeepSeek Reasoning tokens
        "<tool_call>", "</tool_call>",              # Agent tool invocation tokens
        "<tool_response>", "</tool_response>",      # Agent tool response tokens
        "<image>", "</image>",                      # Vision patch anchor tokens
    ]
}
num_added = tokenizer.add_special_tokens(SPECIAL_TOKENS)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

TOTAL_VOCAB_SIZE = max(len(tokenizer), tokenizer.vocab_size)
print(f"✓ Tokenizer ready with {num_added} special tokens | Total Vocab Size: {TOTAL_VOCAB_SIZE:,}")

# 3. Universal Domain Classifier & Adaptive Prompt Formatter
def detect_domain(cols: set, row: Dict[str, Any]) -> str:
    if TASK_DOMAIN != "auto-detect":
        return TASK_DOMAIN

    col_names = " ".join(cols).lower()
    if any(k in col_names for k in ["code", "python", "solution_code", "func", "programming"]):
        return "coding"
    if any(k in col_names for k in ["problem", "gsm8k", "math", "proof", "equation", "steps"]):
        return "mathematics_reasoning"
    if any(k in col_names for k in ["dialog", "dialogue", "conversations", "messages", "chat", "turns"]):
        return "general_chat"
    if any(k in col_names for k in ["instruction", "prompt", "question"]) and any(k in col_names for k in ["output", "response", "answer", "completion"]):
        return "general_chat"
    if any(k in col_names for k in ["image", "pixel_values", "caption"]):
        return "vision"
    return "raw_pretrain"

def format_sample(row: Dict[str, Any]) -> str:
    cols = set(row.keys())
    domain = detect_domain(cols, row)

    # A. Multi-turn Dialogue Dataset (Supports messages list-of-dicts, ShareGPT, dialog list-of-strings)
    diag_col = next((c for c in ["messages", "conversations", "dialog", "dialogue", "chat", "turns"] if c in cols), None)
    if diag_col and isinstance(row[diag_col], (list, tuple)):
        turns = row[diag_col]
        if turns:
            formatted_convo = []
            for i, turn in enumerate(turns):
                if isinstance(turn, dict):
                    raw_role = str(turn.get("role") or turn.get("from") or ("user" if i % 2 == 0 else "assistant")).lower()
                    if raw_role in ["human", "user", "input"]:
                        role = "user"
                    elif raw_role in ["gpt", "assistant", "bot", "output", "model"]:
                        role = "assistant"
                    elif raw_role in ["system"]:
                        role = "system"
                    else:
                        role = raw_role
                    content = str(turn.get("content") or turn.get("value") or turn.get("text") or "").strip()
                else:
                    role = "user" if (i % 2 == 0) else "assistant"
                    content = str(turn).strip()

                if content:
                    formatted_convo.append(f"<|im_start|>{role}\n{content}<|im_end|>")
            if formatted_convo:
                return "\n".join(formatted_convo)

    # Find standard Q&A columns
    inst = next((c for c in ["instruction", "prompt", "question", "problem", "query", "input_text"] if c in cols), None)
    out = next((c for c in ["output", "response", "answer", "solution", "completion", "code", "target"] if c in cols), None)
    inp = next((c for c in ["input", "context", "rationale"] if c in cols), None)

    q_text = str(row[inst]).strip() if inst and row[inst] else ""
    a_text = str(row[out]).strip() if out and row[out] else ""
    c_text = f"\nContext:\n{row[inp]}" if inp and row[inp] else ""

    # B. Coding Domain
    if domain == "coding":
        req = q_text or "Write code for this task."
        return f"<|im_start|>user\n{req}{c_text}<|im_end|>\n<|im_start|>assistant\n```python\n{a_text}\n```<|im_end|>"

    # C. Mathematics & Logical Reasoning Domain (DeepSeek <think> reasoning)
    elif domain == "mathematics_reasoning":
        req = q_text or "Solve this mathematical reasoning problem."
        return f"<|im_start|>user\n{req}{c_text}<|im_end|>\n<|im_start|>assistant\n<think>\nAnalyzing mathematical constraints and calculating step-by-step.\n</think>\n{a_text}<|im_end|>"

    # D. Science / STEM Domain
    elif domain == "science_stem":
        req = q_text or "Explain the scientific principles involved."
        return f"<|im_start|>user\n{req}{c_text}<|im_end|>\n<|im_start|>assistant\n{a_text}<|im_end|>"

    # E. General Chat & Instruction
    elif domain == "general_chat":
        if q_text and a_text:
            return f"<|im_start|>user\n{q_text}{c_text}<|im_end|>\n<|im_start|>assistant\n{a_text}<|im_end|>"

    # F. Vision Multi-modal
    elif domain == "vision":
        cap = str(row.get("text") or row.get("caption") or row.get("response") or "A descriptive visual scene.").strip()
        return f"<image> Describe image: {cap}<|im_end|>"

    # G. Raw Continuous Pretraining (Books, Wiki, Stories)
    text_col = next((c for c in ["text", "content", "body", "article", "story"] if c in cols), None)
    if text_col and row[text_col]:
        return f"{str(row[text_col]).strip()}<|im_end|>"

    # H. Universal Fallback
    vals = [str(v).strip() for v in row.values() if v is not None and not isinstance(v, (dict, list))]
    return " ".join(vals) + "<|im_end|>"

# 4. Universal Fault-Tolerant Dataset Ingestion (Handles Parquet, JSONL, Dynamic Splits & Script Redirection)
print(f"\nIngesting dataset via Mode: '{DATASET_SOURCE}' from '{DATASET_PATH}'...")

KNOWN_PARQUET_MIRRORS = {
    "daily_dialog": "OpenRL/daily_dialog",
    "akhil391/daily_dialog": "OpenRL/daily_dialog",
    "li2017dailydialog/daily_dialog": "OpenRL/daily_dialog",
    "roskon/dailydialog": "OpenRL/daily_dialog",
}

def load_any_dataset(path_or_name: str, source_type: str):
    # A. Local files / Google Drive files
    if source_type == "Custom Upload / Drive" or os.path.exists(path_or_name):
        ext = os.path.splitext(path_or_name)[-1].lower()
        type_map = {".json": "json", ".jsonl": "json", ".csv": "csv", ".tsv": "csv", ".parquet": "parquet", ".txt": "text"}
        file_type = type_map.get(ext, "text")
        try:
            return load_dataset(file_type, data_files=path_or_name, split="train")
        except Exception:
            ds_dict = load_dataset(file_type, data_files=path_or_name)
            return ds_dict[list(ds_dict.keys())[0]]

    # B. Auto-redirect known legacy script repositories to modern canonical Parquet equivalents
    clean_key = path_or_name.lower().strip()
    target_repo = KNOWN_PARQUET_MIRRORS.get(clean_key, path_or_name)
    if target_repo != path_or_name:
        print(f"✓ Detected legacy script repository '{path_or_name}'. Auto-redirecting to canonical Parquet dataset: '{target_repo}'")

    # C. Attempt standard Parquet / JSONL / Hub load (without deprecated trust_remote_code)
    try:
        return load_dataset(target_repo, split="train")
    except Exception as e:
        err_msg = str(e)

        # 1. Handle Hugging Face 'Dataset scripts are no longer supported' error (datasets >= 3.0)
        if "dataset scripts are no longer supported" in err_msg.lower() or "not supported anymore" in err_msg.lower():
            if any(k in clean_key for k in ["daily_dialog", "dailydialog"]):
                print(f"✓ Legacy script blocked by Hugging Face. Auto-redirecting to canonical Parquet dataset: 'OpenRL/daily_dialog'")
                return load_dataset("OpenRL/daily_dialog", split="train")

            # Search Hub for community Parquet version
            try:
                from huggingface_hub import HfApi
                api = HfApi()
                base_name = path_or_name.split("/")[-1]
                matches = list(api.list_datasets(search=base_name, limit=5))
                for m in matches:
                    if m.id != path_or_name:
                        try:
                            print(f"✓ Found Parquet alternative on Hub: '{m.id}'. Loading...")
                            return load_dataset(m.id, split="train")
                        except Exception:
                            continue
            except Exception:
                pass

            raise RuntimeError(
                f"The dataset repository '{path_or_name}' relies on a legacy Python loading script (.py) which modern "
                f"Hugging Face 'datasets >= 3.0' no longer executes for security reasons.\n"
                f"Solution: Please specify a standard Parquet or JSONL dataset repository on Hugging Face."
            ) from e

        # 2. Check multi-subset configs (e.g. glue, wikitext)
        if "config" in err_msg.lower() or "builder" in err_msg.lower():
            try:
                from datasets import get_dataset_config_names
                cfgs = get_dataset_config_names(target_repo)
                if cfgs:
                    print(f"✓ Auto-selected dataset subset: '{cfgs[0]}'")
                    return load_dataset(target_repo, cfgs[0], split="train")
            except Exception:
                pass

        # 3. Inspect all available splits if 'train' is missing (e.g. 'train_sft', 'data', 'validation')
        try:
            ds_dict = load_dataset(target_repo)
            if hasattr(ds_dict, "keys"):
                available_splits = list(ds_dict.keys())
                chosen_split = next((s for s in available_splits if "train" in s.lower()), None)
                if not chosen_split:
                    chosen_split = next((s for s in available_splits if any(k in s.lower() for k in ["sft", "data", "prompt", "chat"])), None)
                if not chosen_split:
                    chosen_split = available_splits[0]
                print(f"✓ Auto-detected split '{chosen_split}' from available: {available_splits}")
                return ds_dict[chosen_split]
        except Exception:
            pass

        raise e

raw_dataset = load_any_dataset(DATASET_PATH, DATASET_SOURCE)

if MAX_SAMPLES is not None and len(raw_dataset) > MAX_SAMPLES:
    raw_dataset = raw_dataset.shuffle(seed=42).select(range(MAX_SAMPLES))

detected_domain = detect_domain(set(raw_dataset[0].keys()), raw_dataset[0])
print(f"✓ Ingested {len(raw_dataset):,} samples | Active Domain: '{detected_domain.upper()}'")
print("\n--- Sample Formatted Input Preview ---")
preview_str = format_sample(raw_dataset[0])
print(preview_str[:300] + ("..." if len(preview_str) > 300 else "") + "\n" + "-" * 38)

def tokenize_batch(examples):
    keys = list(examples.keys())
    batch_len = len(examples[keys[0]])
    formatted = []
    for i in range(batch_len):
        row = {k: examples[k][i] for k in keys}
        formatted.append(format_sample(row))

    tokens = tokenizer(
        formatted,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
        return_tensors=None,
    )
    return {
        "input_ids": tokens["input_ids"],
        "attention_mask": tokens["attention_mask"],
    }

tokenized_dataset = raw_dataset.map(
    tokenize_batch,
    batched=True,
    batch_size=1000,
    remove_columns=raw_dataset.column_names,
    desc="Tokenizing for BabyFlash Causal LM",
)
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])
print(f"✓ Pipeline tokenized {len(tokenized_dataset):,} samples. Tensor shape: {tokenized_dataset[0]['input_ids'].shape}")

In [ ]:
# @title 🧠 Cell 3: Native DeepSeek Architecture Engine (100% GGUF / deepseek2 Aligned)
Run_This_Cell= "3" # @param {type:"string"}
import math
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional, Tuple, List, Dict, Union
from transformers.configuration_utils import PretrainedConfig
from transformers.modeling_utils import PreTrainedModel
from transformers.modeling_outputs import CausalLMOutputWithPast

# 1. Standardized DeepSeek Architecture Scale Configurations
SCALE_CONFIGS = {
    "Baby-Nano": {
        "hidden_size": 96,
        "num_hidden_layers": 3,
        "num_attention_heads": 4,
        "kv_lora_rank": 24,
        "qk_rope_head_dim": 12,
        "intermediate_size": 192,
        "n_routed_experts": 4,
        "num_experts_per_tok": 1,
        "n_shared_experts": 1,
    },
    "Baby-Tiny": {
        "hidden_size": 192,
        "num_hidden_layers": 6,
        "num_attention_heads": 6,
        "kv_lora_rank": 48,
        "qk_rope_head_dim": 16,
        "intermediate_size": 384,
        "n_routed_experts": 8,
        "num_experts_per_tok": 2,
        "n_shared_experts": 1,
    },
    "Baby-Flash": {
        "hidden_size": 256,
        "num_hidden_layers": 8,
        "num_attention_heads": 8,
        "kv_lora_rank": 64,
        "qk_rope_head_dim": 16,
        "intermediate_size": 512,
        "n_routed_experts": 8,
        "num_experts_per_tok": 2,
        "n_shared_experts": 1,
    },
    "Baby-Pro": {
        "hidden_size": 512,
        "num_hidden_layers": 14,
        "num_attention_heads": 8,
        "kv_lora_rank": 128,
        "qk_rope_head_dim": 32,
        "intermediate_size": 1024,
        "n_routed_experts": 16,
        "num_experts_per_tok": 2,
        "n_shared_experts": 2,
    },
    "Custom": {
        "hidden_size": CUSTOM_HIDDEN_SIZE,
        "num_hidden_layers": CUSTOM_NUM_LAYERS,
        "num_attention_heads": 8,
        "kv_lora_rank": CUSTOM_HIDDEN_SIZE // 4,
        "qk_rope_head_dim": 16,
        "intermediate_size": CUSTOM_HIDDEN_SIZE * 2,
        "n_routed_experts": CUSTOM_NUM_EXPERTS,
        "num_experts_per_tok": 2,
        "n_shared_experts": 1,
    }
}
ACTIVE_SCALE = SCALE_CONFIGS.get(MODEL_SCALE, SCALE_CONFIGS["Baby-Flash"])

# Standard DeepSeek-V2/V3 Configuration conforming to llama.cpp's deepseek2 specification
class Deepseek2Config(PretrainedConfig):
    model_type = "deepseek2"
    def __init__(
        self,
        vocab_size: int = 151665,
        hidden_size: int = ACTIVE_SCALE["hidden_size"],
        num_hidden_layers: int = ACTIVE_SCALE["num_hidden_layers"],
        num_attention_heads: int = ACTIVE_SCALE["num_attention_heads"],
        kv_lora_rank: int = ACTIVE_SCALE["kv_lora_rank"],
        qk_rope_head_dim: int = ACTIVE_SCALE["qk_rope_head_dim"],
        intermediate_size: int = ACTIVE_SCALE["intermediate_size"],
        n_routed_experts: int = ACTIVE_SCALE["n_routed_experts"],
        num_experts_per_tok: int = ACTIVE_SCALE["num_experts_per_tok"],
        n_shared_experts: int = ACTIVE_SCALE["n_shared_experts"],
        routed_scaling_factor: float = 1.0,
        router_aux_loss_coef: float = 0.01,
        max_position_embeddings: int = 2048,
        initializer_range: float = 0.02,
        rms_norm_eps: float = 1e-6,
        use_cache: bool = True,
        pad_token_id: int = 151643,
        bos_token_id: int = 151643,
        eos_token_id: int = 151643,
        tie_word_embeddings: bool = True,
        **kwargs,
    ):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.kv_lora_rank = kv_lora_rank
        self.qk_rope_head_dim = qk_rope_head_dim
        self.intermediate_size = intermediate_size
        self.n_routed_experts = n_routed_experts
        self.num_experts_per_tok = num_experts_per_tok
        self.n_shared_experts = n_shared_experts
        self.routed_scaling_factor = routed_scaling_factor
        self.router_aux_loss_coef = router_aux_loss_coef
        self.max_position_embeddings = max_position_embeddings
        self.initializer_range = initializer_range
        self.rms_norm_eps = rms_norm_eps
        self.use_cache = use_cache
        super().__init__(
            pad_token_id=pad_token_id,
            bos_token_id=bos_token_id,
            eos_token_id=eos_token_id,
            tie_word_embeddings=tie_word_embeddings,
            **kwargs,
        )
        self.auto_map = {
            "AutoConfig": "configuration_deepseek.Deepseek2Config",
            "AutoModelForCausalLM": "modeling_deepseek.Deepseek2ForCausalLM"
        }
        self.architectures = ["Deepseek2ForCausalLM"]

# 2. Normalization & Positional Embeddings
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        variance = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(variance + self.eps) * self.weight

class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, max_position_embeddings: int = 2048, base: float = 10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.max_seq_len_cached = max_position_embeddings
        t = torch.arange(self.max_seq_len_cached, dtype=torch.float32)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin_cached", emb.sin()[None, None, :, :], persistent=False)

    def forward(self, seq_len: int, device: torch.device, offset: int = 0) -> Tuple[torch.Tensor, torch.Tensor]:
        total_len = seq_len + offset
        if total_len > self.max_seq_len_cached:
            t = torch.arange(total_len, device=device, dtype=torch.float32)
            freqs = torch.outer(t, self.inv_freq.to(device))
            emb = torch.cat((freqs, freqs), dim=-1)
            cos = emb.cos()[None, None, :, :]
            sin = emb.sin()[None, None, :, :]
            return cos[:, :, offset:total_len, :], sin[:, :, offset:total_len, :]
        return (
            self.cos_cached[:, :, offset:total_len, :].to(device),
            self.sin_cached[:, :, offset:total_len, :].to(device),
        )

def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_emb(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    return (x * cos) + (rotate_half(x) * sin)

# 3. Multi-Head Latent Attention (MLA) with Standard DeepSeek Naming
class Deepseek2Attention(nn.Module):
    def __init__(self, config: Deepseek2Config, layer_idx: int = 0):
        super().__init__()
        self.config = config
        self.layer_idx = layer_idx
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.head_dim = config.hidden_size // config.num_attention_heads
        self.kv_lora_rank = config.kv_lora_rank
        self.qk_rope_head_dim = config.qk_rope_head_dim

        # Standard DeepSeek MLA Projections
        self.q_proj = nn.Linear(self.hidden_size, self.num_heads * self.head_dim, bias=False)
        self.q_rope_proj = nn.Linear(self.hidden_size, self.num_heads * self.qk_rope_head_dim, bias=False)
        self.kv_down_proj = nn.Linear(self.hidden_size, self.kv_lora_rank, bias=False)
        self.kv_norm = RMSNorm(self.kv_lora_rank, eps=config.rms_norm_eps)
        self.k_up_proj = nn.Linear(self.kv_lora_rank, self.num_heads * self.head_dim, bias=False)
        self.v_up_proj = nn.Linear(self.kv_lora_rank, self.num_heads * self.head_dim, bias=False)
        self.k_rope_proj = nn.Linear(self.hidden_size, self.qk_rope_head_dim, bias=False)
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, self.hidden_size, bias=False)
        self.rotary = RotaryEmbedding(self.qk_rope_head_dim, max_position_embeddings=config.max_position_embeddings)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        past_key_value: Optional[Dict[str, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[Dict[str, torch.Tensor]]]:
        B, T, _ = hidden_states.shape
        q_c = self.q_proj(hidden_states).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        q_r = self.q_rope_proj(hidden_states).view(B, T, self.num_heads, self.qk_rope_head_dim).transpose(1, 2)

        c_kv = self.kv_norm(self.kv_down_proj(hidden_states))
        k_r = self.k_rope_proj(hidden_states).view(B, T, 1, self.qk_rope_head_dim).transpose(1, 2)

        offset = 0
        if past_key_value is not None and "c_kv" in past_key_value:
            offset = past_key_value["c_kv"].shape[1]
            c_kv = torch.cat([past_key_value["c_kv"], c_kv], dim=1)
            k_r = torch.cat([past_key_value["k_rope"], k_r], dim=2)

        present_key_value = {"c_kv": c_kv, "k_rope": k_r} if use_cache else None
        total_len = c_kv.shape[1]

        cos, sin = self.rotary(total_len, device=hidden_states.device)
        q_r = apply_rotary_emb(q_r, cos[:, :, offset : offset + T, :], sin[:, :, offset : offset + T, :])
        k_r_rotated = apply_rotary_emb(k_r, cos, sin)

        k_c = self.k_up_proj(c_kv).view(B, total_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_up_proj(c_kv).view(B, total_len, self.num_heads, self.head_dim).transpose(1, 2)

        k_r_expanded = k_r_rotated.repeat(1, self.num_heads, 1, 1)
        q = torch.cat([q_c, q_r], dim=-1)
        k = torch.cat([k_c, k_r_expanded], dim=-1)

        is_causal = (T > 1 and past_key_value is None and attention_mask is None)
        attn_mask = None
        if attention_mask is not None and attention_mask.dim() == 2:
            attn_mask = attention_mask[:, None, None, :].to(dtype=q.dtype)
            attn_mask = (1.0 - attn_mask) * -10000.0

        attn_out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, is_causal=is_causal)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, self.num_heads * self.head_dim)
        return self.o_proj(attn_out), present_key_value

# 4. Standard SwiGLU Expert
class Deepseek2MLP(nn.Module):
    def __init__(self, config: Deepseek2Config, intermediate_size: Optional[int] = None):
        super().__init__()
        inter_dim = intermediate_size or config.intermediate_size
        self.gate_proj = nn.Linear(config.hidden_size, inter_dim, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, inter_dim, bias=False)
        self.down_proj = nn.Linear(inter_dim, config.hidden_size, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

# 5. Native DeepSeekMoE Layer (Standard deepseek2 C++ Layout)
class Deepseek2MoE(nn.Module):
    def __init__(self, config: Deepseek2Config):
        super().__init__()
        self.config = config
        self.n_routed_experts = config.n_routed_experts
        self.top_k = min(config.num_experts_per_tok, self.n_routed_experts)

        # Standard router gate
        self.gate = nn.Linear(config.hidden_size, self.n_routed_experts, bias=False)

        # Routed experts list
        self.experts = nn.ModuleList([
            Deepseek2MLP(config) for _ in range(self.n_routed_experts)
        ])

        # Shared foundation expert (running in parallel with routed experts)
        if config.n_shared_experts > 0:
            self.shared_experts = Deepseek2MLP(config, intermediate_size=config.intermediate_size * config.n_shared_experts)
        else:
            self.shared_experts = None

        self.aux_coef = config.router_aux_loss_coef

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        B, T, C = x.shape
        x_flat = x.view(-1, C)
        router_logits = self.gate(x_flat)
        router_probs = F.softmax(router_logits, dim=-1)

        weights, indices = torch.topk(router_probs, self.top_k, dim=-1)
        weights = weights / (weights.sum(dim=-1, keepdim=True) + 1e-6)

        if self.training and self.aux_coef > 0:
            mask = torch.zeros_like(router_probs).scatter_(1, indices, 1.0)
            f_avg = mask.mean(dim=0)
            P_avg = router_probs.mean(dim=0)
            aux_loss = self.aux_coef * self.n_routed_experts * (P_avg * f_avg).sum()
        else:
            aux_loss = torch.tensor(0.0, device=x.device, dtype=x.dtype)

        routed_out = torch.zeros_like(x_flat)
        for exp_id, expert in enumerate(self.experts):
            for k_idx in range(self.top_k):
                sel = (indices[:, k_idx] == exp_id)
                if sel.any():
                    t_idx = torch.nonzero(sel).squeeze(-1)
                    w = weights[sel, k_idx].unsqueeze(-1)
                    exp_out = expert(x_flat[sel])
                    routed_out.index_put_((t_idx,), exp_out * w, accumulate=True)

        if self.shared_experts is not None:
            shared_out = self.shared_experts(x_flat)
        else:
            shared_out = torch.zeros_like(x_flat)

        return (routed_out + shared_out).view(B, T, C), aux_loss

# 6. DeepSeek-V2/V3 Transformer Decoder Layer
class Deepseek2DecoderLayer(nn.Module):
    def __init__(self, config: Deepseek2Config, layer_idx: int):
        super().__init__()
        self.input_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.self_attn = Deepseek2Attention(config, layer_idx=layer_idx)
        self.post_attention_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.mlp = Deepseek2MoE(config)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        past_key_value: Optional[Dict[str, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, torch.Tensor, Optional[Dict[str, torch.Tensor]]]:
        residual = hidden_states
        normed_h = self.input_layernorm(hidden_states)
        attn_out, present_kv = self.self_attn(normed_h, attention_mask=attention_mask, past_key_value=past_key_value, use_cache=use_cache)
        hidden_states = residual + attn_out

        residual = hidden_states
        normed_h = self.post_attention_layernorm(hidden_states)
        mlp_out, aux_loss = self.mlp(normed_h)
        hidden_states = residual + mlp_out

        return hidden_states, aux_loss, present_kv

# 7. Native DeepSeek Backbone Model
@dataclass
class Deepseek2Output(CausalLMOutputWithPast):
    loss: Optional[torch.FloatTensor] = None
    logits: torch.FloatTensor = None
    past_key_values: Optional[List[Dict[str, torch.Tensor]]] = None
    router_aux_loss: Optional[torch.FloatTensor] = None

class Deepseek2Model(nn.Module):
    def __init__(self, config: Deepseek2Config):
        super().__init__()
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList([
            Deepseek2DecoderLayer(config, i) for i in range(config.num_hidden_layers)
        ])
        self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        past_key_values: Optional[List[Dict[str, torch.Tensor]]] = None,
        use_cache: bool = True,
    ):
        h = self.embed_tokens(input_ids)
        total_aux = torch.tensor(0.0, device=h.device, dtype=h.dtype)
        present_kvs = [] if use_cache else None

        for idx, layer in enumerate(self.layers):
            p_kv = past_key_values[idx] if past_key_values is not None else None
            h, aux_loss, cur_kv = layer(h, attention_mask=attention_mask, past_key_value=p_kv, use_cache=use_cache)
            total_aux = total_aux + aux_loss
            if use_cache:
                present_kvs.append(cur_kv)

        return self.norm(h), total_aux, present_kvs

# 8. Complete Deepseek2ForCausalLM
class Deepseek2ForCausalLM(PreTrainedModel):
    config_class = Deepseek2Config
    base_model_prefix = "model"
    _tied_weights_keys = ["lm_head.weight"]

    def __init__(self, config: Deepseek2Config):
        super().__init__(config)
        self.model = Deepseek2Model(config)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        if config.tie_word_embeddings:
            self.lm_head.weight = self.model.embed_tokens.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
            if module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.Embedding):
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)

    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        past_key_values: Optional[List[Dict[str, torch.Tensor]]] = None,
        labels: Optional[torch.LongTensor] = None,
        use_cache: bool = True,
        return_dict: bool = True,
        **kwargs,
    ):
        h, aux_loss, present_kvs = self.model(input_ids, attention_mask, past_key_values, use_cache)
        logits = self.lm_head(h)

        loss = None
        if labels is not None:
            s_logits = logits[..., :-1, :].contiguous()
            s_labels = labels[..., 1:].contiguous()
            if s_labels.shape[1] < s_logits.shape[1]:
                s_logits = s_logits[:, s_logits.shape[1] - s_labels.shape[1]:, :]
            lm_loss = F.cross_entropy(s_logits.view(-1, self.config.vocab_size), s_labels.view(-1), ignore_index=-100)
            loss = lm_loss + aux_loss

        return Deepseek2Output(loss=loss, logits=logits, past_key_values=present_kvs, router_aux_loss=aux_loss)

    @torch.no_grad()
    def generate(
        self,
        input_ids: torch.LongTensor,
        max_new_tokens: int = 60,
        temperature: float = 0.7,
        top_k: int = 50,
        top_p: float = 0.9,
        repetition_penalty: float = 1.25,
        eos_token_id: Optional[int] = None,
    ) -> torch.LongTensor:
        self.eval()
        eos_ids = set()
        if eos_token_id is not None:
            eos_ids.add(eos_token_id)
        if self.config.eos_token_id is not None:
            eos_ids.add(self.config.eos_token_id)
        im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
        if im_end_id is not None and im_end_id != tokenizer.unk_token_id:
            eos_ids.add(im_end_id)

        gen = input_ids.clone()
        p_kvs = None
        curr_ids = input_ids

        for _ in range(max_new_tokens):
            out = self(input_ids=curr_ids, past_key_values=p_kvs, use_cache=True)
            logits = out.logits[:, -1, :].clone()
            p_kvs = out.past_key_values

            # Anti-Repetition Penalty Masking
            if repetition_penalty != 1.0:
                for b in range(logits.shape[0]):
                    prev_tokens = set(gen[b].tolist())
                    for t_id in prev_tokens:
                        if logits[b, t_id] > 0:
                            logits[b, t_id] /= repetition_penalty
                        else:
                            logits[b, t_id] *= repetition_penalty

            if temperature > 0:
                logits = logits / max(temperature, 1e-4)
                if top_k > 0:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < v[:, [-1]]] = -float("Inf")
                if 0.0 < top_p < 1.0:
                    s_logits, s_indices = torch.sort(logits, descending=True)
                    cum_probs = torch.cumsum(F.softmax(s_logits, dim=-1), dim=-1)
                    s_mask = cum_probs > top_p
                    s_mask[..., 1:] = s_mask[..., :-1].clone()
                    s_mask[..., 0] = 0
                    mask = s_mask.scatter(1, s_indices, s_mask)
                    logits[mask] = -float("Inf")
                probs = F.softmax(logits, dim=-1)
                next_tok = torch.multinomial(probs, num_samples=1)
            else:
                next_tok = torch.argmax(logits, dim=-1, keepdim=True)

            gen = torch.cat([gen, next_tok], dim=1)
            curr_ids = next_tok
            if any(next_tok.item() == eid for eid in eos_ids):
                break

        return gen

# 9. Instantiate Standardized DeepSeek Model
config = Deepseek2Config(
    vocab_size=TOTAL_VOCAB_SIZE,
    pad_token_id=tokenizer.pad_token_id,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)
model = Deepseek2ForCausalLM(config).to(device)

total_params = sum(p.numel() for p in model.parameters())
active_params = sum(p.numel() for n, p in model.named_parameters() if not n.startswith("model.layers") or "shared" in n or "experts.0" in n or "experts.1" in n or "self_attn" in n)
print(f"✓ DeepSeek Native Engine compiled successfully ({MODEL_SCALE}).")
print(f"✓ Total Parameters: {total_params:,} | Active Parameters per Token: ~{active_params:,}")
print(f"✓ GGUF / llama.cpp Architecture Standard: 'deepseek2' (Natively recognized)")


In [ ]:
# @title 🚀 Cell 4: Production Training Engine & GGUF-Ready Model Release
Run_This_Cell= "4" # @param {type:"string"}
import gc
import os
import shutil
import glob
import json
from torch.utils.data import DataLoader
from safetensors.torch import save_file

# 1. Setup Optimizer and Mixed-Precision Scaler
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
use_amp = ENABLE_MIXED_PRECISION and (device == "cuda")
if device == "cuda":
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
else:
    scaler = torch.amp.GradScaler("cpu", enabled=False)

train_loader = DataLoader(tokenized_dataset, batch_size=BATCH_SIZE, shuffle=True)
steps_per_epoch = len(train_loader)
total_target_steps = EPOCHS * (steps_per_epoch // max(1, GRAD_ACCUM_STEPS))

# Cosine Learning Rate Scheduler with Warmup
warmup_steps = max(10, int(0.05 * total_target_steps))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_target_steps), eta_min=1e-6)

# 2. Inspect Existing Drive Checkpoints & Auto-Resume Logic
existing_ckpts = []
if os.path.exists(RUN_CHECKPOINT_DIR):
    for d in os.listdir(RUN_CHECKPOINT_DIR):
        if d.startswith("checkpoint-step-"):
            try:
                s_num = int(d.split("-")[-1])
                existing_ckpts.append((s_num, os.path.join(RUN_CHECKPOINT_DIR, d)))
            except ValueError:
                pass
    existing_ckpts.sort(key=lambda x: x[0])

start_step = 0
start_epoch = 0
loss_history = []

if existing_ckpts:
    latest_step, latest_ckpt_dir = existing_ckpts[-1]
    print("=" * 60)
    print(f"[✓] Existing training session found for Scale '{MODEL_SCALE}' on Dataset '{DATASET_PATH}'.")
    print(f"[✓] Resuming seamlessly from checkpoint: {latest_ckpt_dir} (Step {latest_step:,})...")
    print("=" * 60)

    try:
        weight_file = os.path.join(latest_ckpt_dir, "pytorch_model.bin")
        if os.path.exists(weight_file):
            model.load_state_dict(torch.load(weight_file, map_location=device))
        else:
            model = Deepseek2ForCausalLM.from_pretrained(latest_ckpt_dir).to(device)

        state_file = os.path.join(latest_ckpt_dir, "training_state.pt")
        if os.path.exists(state_file):
            state = torch.load(state_file, map_location=device)
            optimizer.load_state_dict(state["optimizer_state_dict"])
            if use_amp and state.get("scaler_state_dict") is not None:
                scaler.load_state_dict(state["scaler_state_dict"])
            start_step = state.get("step", latest_step)
            start_epoch = state.get("epoch", start_step // max(1, steps_per_epoch))
            loss_history = state.get("loss_history", [])

        print(f"✓ Resumed successfully! Continuing from Epoch {start_epoch + 1}, Global Step {start_step:,}...")
    except Exception as e:
        print(f"Notice: Auto-resume encountered issue ({e}). Starting fresh from step 0.")
        start_step = 0
        start_epoch = 0
else:
    print("=" * 60)
    print(f"[+] Starting fresh training run: '{UNIQUE_RUN_ID}' with {total_target_steps:,} optimization steps.")
    print("=" * 60)

# 3. Google Drive Rolling Checkpoint Saver (Resumption State Only, Limit = 3)
def save_drive_checkpoint(curr_step: int, curr_epoch: int, is_emergency: bool = False):
    tag = f"checkpoint-step-{curr_step}" if not is_emergency else f"checkpoint-emergency-step-{curr_step}"
    save_path = os.path.join(RUN_CHECKPOINT_DIR, tag)
    os.makedirs(save_path, exist_ok=True)

    # Save minimal weights for resume
    torch.save(model.state_dict(), os.path.join(save_path, "pytorch_model.bin"))
    config.save_pretrained(save_path)

    # Save Training State (Optimizer, Scaler, Step, Epoch, Loss)
    state = {
        "step": curr_step,
        "epoch": curr_epoch,
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict() if use_amp else None,
        "loss_history": loss_history,
        "unique_run_id": UNIQUE_RUN_ID,
    }
    torch.save(state, os.path.join(save_path, "training_state.pt"))
    status_label = "EMERGENCY" if is_emergency else "SCHEDULED"
    print(f"\n💾 [{status_label} SYNC] Checkpoint synced to Drive: {save_path}")

    # Enforce Rolling Limit (keep top 3 to protect Drive storage)
    all_ckpts = []
    for d in os.listdir(RUN_CHECKPOINT_DIR):
        if d.startswith("checkpoint-step-"):
            try:
                s_num = int(d.split("-")[-1])
                all_ckpts.append((s_num, os.path.join(RUN_CHECKPOINT_DIR, d)))
            except ValueError:
                pass
    all_ckpts.sort(key=lambda x: x[0])

    while len(all_ckpts) > MAX_CHECKPOINTS_TO_KEEP:
        oldest_step, oldest_dir = all_ckpts.pop(0)
        shutil.rmtree(oldest_dir, ignore_errors=True)
        print(f"🧹 [DRIVE OPTIMIZER] Pruned older checkpoint: {oldest_dir}")

# 4. Final Local Package Exporter (Hugging Face Standards + deepseek2 Safetensors)
def export_local_model_package(export_dir: str):
    os.makedirs(export_dir, exist_ok=True)
    print("=" * 60)
    print(f"📦 Packaging Native GGUF-Ready Model into: '{export_dir}'")
    print("=" * 60)

    # A. Save model.safetensors (Cloned to eliminate duplicate shared pointer errors)
    state_dict_safe = {k: v.clone().contiguous() for k, v in model.state_dict().items()}
    safetensors_path = os.path.join(export_dir, "model.safetensors")
    save_file(state_dict_safe, safetensors_path, metadata={"format": "pt"})
    print(f"  ✓ [safetensors] Saved: {safetensors_path}")

    # B. Save legacy pytorch_model.bin
    torch.save(model.state_dict(), os.path.join(export_dir, "pytorch_model.bin"))
    print("  ✓ [weights] Saved: pytorch_model.bin")

    # C. Save config.json (with standard deepseek2 architecture)
    config.save_pretrained(export_dir)
    print("  ✓ [config] Saved: config.json (Architecture: 'deepseek2')")

    # D. Save Tokenizer files
    tokenizer.save_pretrained(export_dir)
    print("  ✓ [tokenizer] Saved: tokenizer.json, tokenizer_config.json, vocab.json")

    # E. Save generation_config.json
    gen_config = {
        "bos_token_id": int(config.bos_token_id or 151643),
        "eos_token_id": int(config.eos_token_id or 151643),
        "pad_token_id": int(config.pad_token_id or 151643),
        "temperature": 0.7,
        "top_p": 0.9,
        "top_k": 50,
        "repetition_penalty": 1.25,
        "max_new_tokens": 128,
        "do_sample": True,
    }
    with open(os.path.join(export_dir, "generation_config.json"), "w", encoding="utf-8") as f:
        json.dump(gen_config, f, indent=2)
    print("  ✓ [generation] Saved: generation_config.json")

    # F. Save Model Card README.md
    model_card = f"""# {MODEL_SCALE} (DeepSeek-V2/V3 Standard Architecture)

An ultra-efficient, native DeepSeekMoE + MLA model fully compatible with **`llama.cpp`**, **LM Studio**, and **Ollama**.

## Architecture Highlights
- **Architecture**: `deepseek2` (Native C++ support in `llama.cpp`)
- **Attention**: Multi-Head Latent Attention (MLA) with low-rank KV compression
- **Sparse Routing**: DeepSeekMoE ({config.n_routed_experts} Routed Experts, Top-{config.num_experts_per_tok} Active + {config.n_shared_experts} Shared Expert)
- **Vocab Size**: {config.vocab_size:,}
- **Hidden Dim**: {config.hidden_size}

## Quickstart (LM Studio / Ollama)
Run directly from Ollama:
```bash
ollama create baby-ai -f Modelfile
ollama run baby-ai
```
"""
    with open(os.path.join(export_dir, "README.md"), "w", encoding="utf-8") as f:
        f.write(model_card)
    print("  ✓ [model_card] Saved: README.md")
    print(f"🎉 Complete GGUF-aligned release package is ready in: {export_dir}")

# 5. Training Loop with Gradient Accumulation & OOM Guardrails
model.train()
global_step = start_step
accum_step = 0

try:
    for epoch in range(start_epoch, EPOCHS):
        epoch_loss, epoch_aux = 0.0, 0.0
        successful_steps = 0
        optimizer.zero_grad()

        for batch_idx, batch in enumerate(train_loader):
            current_batch_global_step = epoch * steps_per_epoch + batch_idx
            if current_batch_global_step < start_step * GRAD_ACCUM_STEPS:
                continue

            try:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)

                labels = input_ids.clone()
                if tokenizer.pad_token_id is not None:
                    labels[labels == tokenizer.pad_token_id] = -100

                autocast_device = "cuda" if device == "cuda" else "cpu"
                with torch.amp.autocast(autocast_device, enabled=use_amp):
                    outputs = model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels,
                    )
                    loss = outputs.loss / GRAD_ACCUM_STEPS

                scaler.scale(loss).backward()
                accum_step += 1

                epoch_loss += loss.item() * GRAD_ACCUM_STEPS
                aux_val = outputs.router_aux_loss.item() if outputs.router_aux_loss is not None else 0.0
                epoch_aux += aux_val

                if accum_step % GRAD_ACCUM_STEPS == 0 or (batch_idx + 1) == len(train_loader):
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
                    scheduler.step()

                    successful_steps += 1
                    global_step += 1
                    loss_history.append((global_step, round(loss.item() * GRAD_ACCUM_STEPS, 4)))

                    if global_step % 50 == 0 or global_step == 1:
                        current_lr = scheduler.get_last_lr()[0]
                        print(f"Epoch [{epoch+1}/{EPOCHS}] Step [{global_step:>5d}/{total_target_steps}] Loss: {loss.item() * GRAD_ACCUM_STEPS:.4f} (MoE Aux: {aux_val:.4f}) | LR: {current_lr:.2e}")

                    if SAVE_TO_DRIVE and (global_step % SAVE_EVERY_N_STEPS == 0):
                        save_drive_checkpoint(global_step, epoch, is_emergency=False)

            except torch.cuda.OutOfMemoryError:
                print(f"⚠️ OOM intercepted at step {global_step}. Purging CUDA cache and resuming...")
                gc.collect()
                torch.cuda.empty_cache()
                optimizer.zero_grad()
                continue

        avg_loss = epoch_loss / max(1, successful_steps * GRAD_ACCUM_STEPS)
        avg_aux = epoch_aux / max(1, successful_steps * GRAD_ACCUM_STEPS)
        print(f"\n>>> Epoch {epoch+1} Complete | Average Loss = {avg_loss:.4f} | MoE Aux = {avg_aux:.4f}\n")

    if SAVE_TO_DRIVE:
        save_drive_checkpoint(global_step, EPOCHS, is_emergency=False)
    export_local_model_package(LOCAL_EXPORT_DIR)
    print(f"🎉 Training fully completed! Final model packaged at: {LOCAL_EXPORT_DIR}")

except KeyboardInterrupt:
    print("\n" + "!" * 60)
    print("⚠️ Training paused by user! Triggering emergency checkpoint...")
    print("!" * 60)
    if SAVE_TO_DRIVE:
        save_drive_checkpoint(global_step, epoch, is_emergency=True)
    export_local_model_package(LOCAL_EXPORT_DIR)
    print(f"✓ State safely preserved. Re-running will resume from step {global_step:,}!")


In [ ]:
# @title 💬 Cell 5: Test, Infer & Native GGUF Exporter (LM Studio & Ollama)
# @markdown Test generation across domains, export standard GGUF binary, or publish to Hugging Face Hub.

TEST_PROMPT = "who are you ?"  # @param {type:"string"}
MAX_NEW_TOKENS = 60  # @param {type:"integer"}
TEMPERATURE = 0.7  # @param {type:"number"}
TOP_P = 0.9  # @param {type:"number"}
TOP_K = 50  # @param {type:"integer"}
REPETITION_PENALTY = 1.25  # @param {type:"number"}

# 1. Interactive Anti-Repetition Inference Function
def test_inference(prompt: str):
    model.eval()
    formatted_prompt = f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"
    print(f"\n[Input Prompt ({detected_domain.upper()})]: {prompt}")
    input_ids = tokenizer(formatted_prompt, return_tensors="pt")["input_ids"].to(device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_k=TOP_K,
            top_p=TOP_P,
            repetition_penalty=REPETITION_PENALTY,
            eos_token_id=tokenizer.eos_token_id,
        )

    full_response = tokenizer.decode(output_ids[0], skip_special_tokens=False)
    assistant_reply = full_response
    if "<|im_start|>assistant\n" in full_response:
        assistant_reply = full_response.split("<|im_start|>assistant\n")[-1]
    if "<|im_end|>" in assistant_reply:
        assistant_reply = assistant_reply.split("<|im_end|>")[0]

    print(f"\n[DeepSeek Output]:\n{assistant_reply.strip()}\n")
    return assistant_reply

# Run Main Interactive Test
test_inference(TEST_PROMPT)

# Specialized Domain Verification
if detected_domain == "coding":
    print("--- Domain Code Generation Test ---")
    test_inference("Write a Python function to check if a string is a palindrome.")

elif detected_domain == "mathematics_reasoning":
    print("--- Domain Mathematical Reasoning Test (<think> tag) ---")
    test_inference("Solve for x: 3x + 12 = 36.")

elif detected_domain == "science_stem":
    print("--- Domain Science / STEM Test ---")
    test_inference("Explain Newton's third law of motion in simple terms.")

# 2. Native GGUF & Ollama Modelfile Exporter for deepseek2 Architecture
def export_native_gguf_and_ollama(model_dir: str):
    print("=" * 60)
    print(f"🛠️ Exporting Native GGUF for LM Studio & Ollama (Architecture: 'deepseek2')...")
    print("=" * 60)

    # A. Generate Ollama Modelfile with Native ChatML and DeepSeek Parameters
    modelfile_path = os.path.join(model_dir, "Modelfile")
    q3 = chr(34) * 3
    modelfile_content = (
        "FROM ./baby_ai.gguf\n"
        "PARAMETER temperature 0.7\n"
        "PARAMETER top_p 0.9\n"
        "PARAMETER repeat_penalty 1.25\n"
        "PARAMETER stop " + chr(34) + "<|im_end|>" + chr(34) + "\n"
        f"TEMPLATE {q3}<|im_start|>user\n"
        "{{ .Prompt }}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{q3}\n"
    )
    with open(modelfile_path, "w", encoding="utf-8") as f:
        f.write(modelfile_content)
    print(f"  ✓ [Ollama] Modelfile created: {modelfile_path}")

    # B. Package Tensors into Standard GGUF Container via python gguf library
    gguf_output_path = os.path.join(model_dir, "baby_ai.gguf")
    try:
        import gguf
        print(f"  ✓ Packaging deepseek2 tensors into GGUF container: {gguf_output_path}...")
        writer = gguf.GGUFWriter(path=gguf_output_path, arch="deepseek2")
        writer.add_name(MODEL_SCALE)
        writer.add_context_length(MAX_SEQ_LENGTH)
        writer.add_embedding_length(config.hidden_size)
        writer.add_block_count(config.num_hidden_layers)
        writer.add_feed_forward_length(config.intermediate_size)
        writer.add_head_count(config.num_attention_heads)

        # Standard DeepSeek MoE & MLA metadata recognized by llama.cpp
        writer.add_uint32("deepseek2.expert_count", config.n_routed_experts)
        writer.add_uint32("deepseek2.expert_used_count", config.num_experts_per_tok)
        writer.add_uint32("deepseek2.expert_shared_count", config.n_shared_experts)
        writer.add_uint32("deepseek2.kv_lora_rank", config.kv_lora_rank)

        for name, param in model.state_dict().items():
            arr = param.cpu().float().numpy()
            writer.add_tensor(name, arr)

        writer.write_header_to_file()
        writer.write_kv_data_to_file()
        writer.write_tensors_to_file()
        writer.close()
        print(f"🎉 GGUF exported successfully: {gguf_output_path}")
        print("  ✓ 100% Natively loadable into LM Studio & Ollama!")
    except ImportError:
        print("  Notice: 'gguf' Python library not loaded. Run '!pip install gguf' to produce the binary .gguf file.")
    except Exception as e:
        print(f"  Notice: GGUF serialization info: {e}")

    print("\n🚀 [How to Run in Ollama on Local Machine]:")
    print(f"   1. Download the '{model_dir}' folder to your computer.")
    print("   2. Run: ollama create baby-ai -f Modelfile")
    print("   3. Run: ollama run baby-ai")
    print("\n🚀 [How to Run in LM Studio]:")
    print(f"   Drag & drop '{os.path.basename(gguf_output_path)}' directly into LM Studio!")

export_native_gguf_and_ollama(LOCAL_EXPORT_DIR)

# 3. 1-Click Hugging Face Hub Publishing
if PUSH_TO_HUB and HF_TOKEN and HF_REPO_ID:
    try:
        from huggingface_hub import HfApi, login
        print(f"\nAuthenticating with Hugging Face Hub...")
        login(token=HF_TOKEN)
        api = HfApi()
        print(f"Uploading full '{LOCAL_EXPORT_DIR}' folder to '{HF_REPO_ID}'...")
        api.upload_folder(
            folder_path=LOCAL_EXPORT_DIR,
            repo_id=HF_REPO_ID,
            repo_type="model",
            token=HF_TOKEN
        )
        print(f"🎉 Successfully published to: https://huggingface.co/{HF_REPO_ID}")
    except Exception as e:
        print(f"Notice: Hub publish encountered: {e}")
elif PUSH_TO_HUB:
    print("\nNotice: Set PUSH_TO_HUB=True with valid HF_TOKEN and HF_REPO_ID in Cell 1 to publish.")
